# Part 5: Multi-source KB + Work IQ + Fabric IQ

Create a fresh KB that combines all lab source categories: restored HR and health indexes, Work IQ, and Fabric Ontology. It is standalone and recreates every knowledge source it needs.

## 1 - Setup

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import requests
from azure.identity import AzureCliCredential
from dotenv import dotenv_values

API_VERSION = "2026-05-01-preview"
TENANT_ID = "72f988bf-86f1-41af-91ab-2d7cd011db47"
ENV_PATH = Path(".").resolve() / ".env"
file_env = dotenv_values(ENV_PATH) if ENV_PATH.exists() else {}
env = {**file_env, **os.environ}


def require_env(name: str) -> str:
    value = env.get(name)
    if not value:
        raise KeyError(f"Set {name} in {ENV_PATH}")
    return value.strip()


ENDPOINT = require_env("AZURE_SEARCH_SERVICE_ENDPOINT").rstrip("/")
API_KEY = require_env("AZURE_SEARCH_ADMIN_KEY")
RAW_CHAT_ENDPOINT = env.get("AZURE_OPENAI_ENDPOINT", "").rstrip("/")

def normalize_chat_endpoint(endpoint: str) -> str:
    if ".openai.azure.com" in endpoint:
        resource_name = endpoint.split("//", 1)[1].split(".", 1)[0]
        return f"https://{resource_name}.services.ai.azure.com"
    return endpoint


CHAT_ENDPOINT = normalize_chat_endpoint(RAW_CHAT_ENDPOINT)
CHAT_KEY = env.get("AZURE_OPENAI_KEY", "")
CHAT_DEPLOYMENT = env.get("AZURE_OPENAI_CHATGPT_DEPLOYMENT", "gpt-5.4-mini")
CHAT_MODEL = env.get("AZURE_OPENAI_CHATGPT_MODEL_NAME", CHAT_DEPLOYMENT)
HRDOCS_INDEX = "hrdocs"
HEALTHDOCS_INDEX = "healthdocs"
stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")


def api_url(path: str) -> str:
    sep = "&" if "?" in path else "?"
    return f"{ENDPOINT}{path}{sep}api-version={API_VERSION}"


search_credential = AzureCliCredential(tenant_id=TENANT_ID)


def get_search_bearer_token() -> str:
    return search_credential.get_token("https://search.azure.com/.default").token


session = requests.Session()
session.headers.update({"api-key": API_KEY, "Accept": "application/json"})


def pp(response: requests.Response, max_chars: int = 5000):
    print(f"{response.status_code} {response.reason}")
    try:
        print(json.dumps(response.json(), indent=2)[:max_chars])
    except ValueError:
        print(response.text[:max_chars])


def expect_success(response: requests.Response, *allowed: int):
    if response.status_code not in allowed:
        raise RuntimeError(f"Expected HTTP {allowed}, got {response.status_code}: {response.text[:1000]}")
    return response


class KnowledgeBaseClient:
    def __init__(self, session: requests.Session):
        self.session = session

    def retrieve(self, knowledge_base_name: str, body: dict, *, headers: dict | None = None, timeout: int = 180):
        return self.session.post(api_url(f"/knowledgebases('{knowledge_base_name}')/retrieve"), json=body, headers=headers or {}, timeout=timeout)


kb_client = KnowledgeBaseClient(session)


def create_search_index_source(name: str, index_name: str, description: str):
    body = {
        "name": name,
        "description": description,
        "kind": "searchIndex",
        "searchIndexParameters": {
            "searchIndexName": index_name,
            "sourceDataFields": [{"name": "uid"}, {"name": "snippet_parent_id"}, {"name": "blob_path"}, {"name": "snippet"}],
            "searchFields": [{"name": "snippet"}],
            "semanticConfigurationName": "semantic-configuration",
        },
    }
    r = session.put(api_url(f"/knowledgesources('{name}')"), json=body, headers={"Prefer": "return=representation"})
    pp(r)
    expect_success(r, 200, 201)
    return body


def azure_openai_model():
    if not CHAT_ENDPOINT or not CHAT_KEY:
        raise RuntimeError("Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_KEY in .env for KBs that use chat models.")
    return {"kind": "azureOpenAI", "azureOpenAIParameters": {"resourceUri": CHAT_ENDPOINT + "/", "apiKey": CHAT_KEY, "deploymentId": CHAT_DEPLOYMENT, "modelName": CHAT_MODEL}}


def create_knowledge_base(name: str, description: str, source_names: list[str], *, output_mode: str = "extractiveData", reasoning: str = "low", include_model: bool = False, instructions: str | None = None):
    body = {"name": name, "description": description, "outputMode": output_mode, "retrievalReasoningEffort": {"kind": reasoning}, "knowledgeSources": [{"name": n} for n in source_names]}
    if include_model:
        body["models"] = [azure_openai_model()]
    if instructions:
        body["retrievalInstructions"] = instructions
    r = session.put(api_url(f"/knowledgebases('{name}')"), json=body, headers={"Prefer": "return=representation"})
    pp(r)
    expect_success(r, 200, 201)
    return body


def search_index_params(source_names: list[str], *, always_query: bool = True) -> list[dict]:
    return [{"kind": "searchIndex", "knowledgeSourceName": n, "includeReferences": True, "includeReferenceSourceData": True, "alwaysQuerySource": always_query} for n in source_names]


def print_copilot_cli_sidequest(knowledge_base_name: str):
    mcp_url = f"{ENDPOINT}/knowledgebases/{knowledge_base_name}/mcp?api-version={API_VERSION}"
    headers = {"api-key": API_KEY}
    print("Service key for the MCP api-key header:")
    print(API_KEY)
    config = {"mcpServers": {f"lab532-{knowledge_base_name}": {"type": "http", "url": mcp_url, "headers": headers}}}
    print("Open notebooks/copilot-cli-sidequest.md for the sidequest steps.")
    print("KB MCP URL:")
    print(mcp_url)
    print("MCP config snippet:")
    print(json.dumps(config, indent=2))


def cleanup_resources(*, kb_names: list[str], ks_names: list[str]):
    for kb_name in kb_names:
        r = session.delete(api_url(f"/knowledgebases('{kb_name}')"))
        print(f"Delete KB {kb_name}: {r.status_code} {r.reason}")
    for ks_name in ks_names:
        r = session.delete(api_url(f"/knowledgesources('{ks_name}')"))
        print(f"Delete KS {ks_name}: {r.status_code} {r.reason}")


print(f"Search endpoint : {ENDPOINT}")
print("Search auth     : api-key")
print(f"API version     : {API_VERSION}")
print(f"Timestamp stamp : {stamp}")
FABRIC_ENDPOINT = env.get("FABRIC_ENDPOINT", "https://api.fabric.microsoft.com")
FABRIC_WORKSPACE_ID = env.get("FABRIC_WORKSPACE_ID", "b0a49ce1-0af9-4d14-a6b6-f6509b0aeec8")
FABRIC_ONTOLOGY_ID = env.get("FABRIC_ONTOLOGY_ID", "cee04244-7307-4266-b760-31c372f98550")
print(f"Fabric endpoint : {FABRIC_ENDPOINT}")
print(f"Workspace ID    : {FABRIC_WORKSPACE_ID}")
print(f"Ontology ID     : {FABRIC_ONTOLOGY_ID}")

## 2 - Get the query-source token

In [ ]:
QUERY_SOURCE_AUTH = get_search_bearer_token()
print(f"Query-source token obtained ({len(QUERY_SOURCE_AUTH)} chars)")

## 3 - Create all knowledge sources

In [ ]:
HR_KS_NAME = f"lab532-hrdocs-{stamp}"
HEALTH_KS_NAME = f"lab532-healthdocs-{stamp}"
WORK_KS_NAME = f"lab532-workiq-{stamp}"
FABRIC_KS_NAME = f"lab532-fabric-ontology-{stamp}"
KS_NAMES = [HR_KS_NAME, HEALTH_KS_NAME, WORK_KS_NAME, FABRIC_KS_NAME]
create_search_index_source(HR_KS_NAME, HRDOCS_INDEX, "LAB532 HR documents")
create_search_index_source(HEALTH_KS_NAME, HEALTHDOCS_INDEX, "LAB532 health benefits documents")
work_ks_body = {"name": WORK_KS_NAME, "kind": "workIQ", "description": "LAB532 Work IQ knowledge source"}
r = session.put(api_url(f"/knowledgesources('{WORK_KS_NAME}')"), json=work_ks_body, headers={"Prefer":"return=representation"})
pp(r)
expect_success(r, 200, 201)
fabric_ks_body = {"name": FABRIC_KS_NAME, "kind": "fabricOntology", "description": "LAB532 Fabric Ontology knowledge source", "fabricOntologyParameters": {"fabricEndpoint": FABRIC_ENDPOINT, "workspaceId": FABRIC_WORKSPACE_ID, "ontologyId": FABRIC_ONTOLOGY_ID}}
r = session.put(api_url(f"/knowledgesources('{FABRIC_KS_NAME}')"), json=fabric_ks_body, headers={"Prefer":"return=representation"})
pp(r)
expect_success(r, 200, 201)

## 4 - Create the combined KB

In [ ]:
KB_NAME = f"lab532-work-fabric-kb-{stamp}"
create_knowledge_base(KB_NAME, "LAB532 KB combining restored indexes, Work IQ, and Fabric Ontology", KS_NAMES, reasoning="low", include_model=True, instructions="Use Work IQ for workplace context, Fabric Ontology for structured operational data, and restored indexes for HR and health policy documents.")

## 5 - `kb_client.retrieve`: ask the combined KB

Combined Work IQ + Fabric IQ retrieves may return `206 Partial Content` when one source succeeds and another source is partially unavailable. Treat `200` as full success and `206` as useful partial success with details in the activity payload.

In [ ]:
retrieve_body = {"includeActivity": True, "messages": [{"role":"user", "content":[{"type":"text", "text":"Use Work IQ for any relevant context about Gresham Insurance Company Limited and Aviva International Insurance. Use the Zava DIY Fabric ontology Product entity to list a few product names with category and stockLevel, then explain how stockLevel can help with inventory planning."}]}], "knowledgeSourceParams": [*search_index_params([HR_KS_NAME, HEALTH_KS_NAME]), {"kind":"workIQ", "knowledgeSourceName":WORK_KS_NAME, "includeReferences":True, "includeReferenceSourceData":True}, {"kind":"fabricOntology", "knowledgeSourceName":FABRIC_KS_NAME, "includeReferences":True, "includeReferenceSourceData":True}], "maxRuntimeInSeconds": 300}
r = kb_client.retrieve(KB_NAME, retrieve_body, headers={"x-ms-query-source-authorization": QUERY_SOURCE_AUTH}, timeout=300)
pp(r)
expect_success(r, 200, 206)

## 6 - Copilot CLI sidequest checkpoint

Use this combined KB in the [Copilot CLI sidequest](copilot-cli-sidequest.md).

In [ ]:
print_copilot_cli_sidequest(KB_NAME)

## 7 - Cleanup

In [ ]:
cleanup_resources(kb_names=[KB_NAME], ks_names=KS_NAMES)

You completed the refreshed LAB532 notebook sequence.